In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS de_mini_project.silver")

In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.silver.customer")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.silver.customer (
    customer_id STRING,
    name STRING,
    contact_info STRING,
    joined_date DATE,
    gender STRING    
)
""")

In [0]:
spark.sql("""
INSERT INTO de_mini_project.silver.customer (customer_id, name, contact_info, joined_date, gender)
WITH cleaned_data AS (
    SELECT 
        _cust_ref_id_ AS customer_id,
        INITCAP(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(REPLACE(full_name, '_', ' '), '["\\\\.]', ''), '\\\\(.*?\\\\)', ''))) AS name,
        NULLIF(REGEXP_REPLACE(_contact_info_, '[^0-9]', ''), '') AS contact_info,
        COALESCE(
            try_to_date(joined_date_, 'dd-MM-yyyy'),
            try_to_date(joined_date_, 'MMM dd, yyyy'),
            try_to_date(joined_date_, 'yyyy.MM.dd')
        ) AS joined_date,
        CASE WHEN gender_code = 'M' THEN 'Male'
             WHEN gender_code = 'F' THEN 'Female'
             ELSE 'Unknown'
        END AS gender
        
    FROM de_mini_project.azure_blob_storage.customers
)
SELECT 
    customer_id, 
    name, 
    contact_info, 
    joined_date, 
    gender
FROM cleaned_data
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY joined_date DESC) = 1
""")

In [0]:
display(spark.sql("SELECT * FROM de_mini_project.silver.customer"))